# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is referenced via a Croissant schema URL and is structured according to the [FAIR principles](https://www.go-fair.org/fair-principles/).

In [ ]:
# Ensure `mlcroissant` is installed (restart kernel after install if needed)
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Set the Croissant schema URL (as provided)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
dataset = mlc.Dataset(croissant_url)

# The dataset's metadata as a Python object
metadata_obj = dataset.metadata
print(f"Dataset name: {metadata_obj.name}\nDescription: {metadata_obj.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We list the available record sets (tables) and their IDs. For each, we also display fields and columns along with their `@id`s for later referencing.

In [ ]:
# List all record sets available in the dataset, using their @id.
print("List of record sets (by @id):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"@id: {rs._id}, Name: {rs.name if hasattr(rs, 'name') else ''}")

# Let's inspect the first record set's fields and columns by @id
if record_sets:
    print("\nFields and columns for the first record set:")
    record_set = record_sets[0]
    # List fields
    if hasattr(record_set, 'fields'):
        print("Fields:")
        for f in record_set.fields:
            print(f"    Field @id: {f._id}, Name: {getattr(f, 'name', '')}, Data type: {getattr(f, 'dataType', '')}")
    # List columns
    if hasattr(record_set, 'columns'):
        print("Columns:")
        for c in record_set.columns:
            print(f"    Column @id: {c._id}, Name: {getattr(c, 'name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract all rows from each available record set into individual Pandas DataFrames, keyed by their `@id`.

In [ ]:
# Extract data from each record set using their @id
dataframes = {}
for rs in dataset.record_sets:
    print(f"Loading records for record set @id: {rs._id}")
    records = list(dataset.records(record_set=rs._id))
    dataframes[rs._id] = pd.DataFrame(records)
    print(f"  Columns: {dataframes[rs._id].columns.tolist()}")
    print(f"  Sample rows:")
    display(dataframes[rs._id].head())

# Choose a record set for downstream analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0] # pick the first one for demonstration
    print(f"We'll use record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps: filtering, normalization, categorization, and grouping. All fields should be referenced by their `@id` in operations.

Below, we select a numerical field (by its `@id`) for demonstration, and show basic filtering and normalization steps.

In [ ]:
df = dataframes[main_record_set_id]

# List numeric-like columns by inspecting their types
print(f"\nAvailable DataFrame columns for EDA:")
for col in df.columns:
    col_values = df[col].dropna()
    if len(col_values) > 0:
        try:
            # Try conversion to float
            pd.to_numeric(col_values.iloc[:5])
            print(f"  {col}: numeric example values: {col_values.iloc[:5].tolist()}")
        except Exception:
            print(f"  {col}: non-numeric / text example. Sample: {col_values.iloc[:5].tolist()}")

# For demonstration, select the first numeric column
numeric_field = None
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
        df[[col]].dropna().mean() # will succeed for numeric
        numeric_field = col
        break
    except:
        continue

if numeric_field is not None:
    print(f"\nSelected numeric_field for EDA (using @id): {numeric_field}")
    # Example: filter values > threshold
    threshold = df[numeric_field].mean() # As an example, filter above average
    filtered_df = df[df[numeric_field] > threshold].copy()

    print(f"Filtered records where {numeric_field} > {threshold:.2f} (mean value):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Pick a categorical/text field for grouping (first non-numeric one)
    group_field = None
    for col in df.columns:
        if col != numeric_field:
            try:
                pd.to_numeric(df[col])
                # Numeric column, skip
                continue
            except:
                group_field = col
                break
    if group_field:
        print(f"\nGrouping by field @id: {group_field}\n")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean','count'])
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize a numeric field's distribution and, if possible, a boxplot grouped by a categorical field.

**Note:** All fields are referenced by their `@id` as required.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} (@id)')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If group_field is not None and has reasonable cardinality, show boxplot
    if group_field is not None and df[group_field].nunique() <= 20:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f'{numeric_field} grouped by {group_field} (@id)')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
We demonstrated how to load a Croissant-FAIR dataset, enumerate record sets, extract and analyze data fields, and visualize key distributions using the mlcroissant library. All elements are referenced by their Croissant `@id`, ensuring reproducible and schema-driven analysis.

Next steps could involve:
- Joining record sets via shared keys (`@id`),
- Advanced feature engineering,
- Downstream ML or analysis workflows.

Refer to the [Croissant specification](https://mlcommons.org/croissant/) and [mlcroissant documentation](https://github.com/mlcommons/croissant) for more advanced functionalities.